# 🤖 03 — Modélisation
**Objectif** : Entraîner et comparer 3 modèles, évaluer avec AUC, classification report et courbe ROC.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, roc_curve, ConfusionMatrixDisplay, confusion_matrix
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

df_enc = pd.read_csv(r'C:\Users\imene\churn-prediction\data\telco_encoded.csv')
print(f"Dataset chargé : {df_enc.shape}")

## 1. Préparation des données

In [ ]:
X = df_enc.drop('Churn', axis=1)
y = df_enc['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Rééquilibrage SMOTE
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

# Normalisation
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_bal)
X_test_sc = scaler.transform(X_test)

print(f"Train : {X_train_sc.shape} | Test : {X_test_sc.shape}")
print(f"Après SMOTE — distribution train : {pd.Series(y_train_bal).value_counts().to_dict()}")

## 2. Entraînement et comparaison des modèles

In [ ]:
modeles = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42)
}

resultats = {}
probas = {}

for nom, modele in modeles.items():
    modele.fit(X_train_sc, y_train_bal)
    y_proba = modele.predict_proba(X_test_sc)[:, 1]
    y_pred = modele.predict(X_test_sc)
    auc = roc_auc_score(y_test, y_proba)
    cv = cross_val_score(modele, X_train_sc, y_train_bal, cv=5, scoring='roc_auc')
    resultats[nom] = {'auc': auc, 'cv_mean': cv.mean(), 'cv_std': cv.std(), 'y_pred': y_pred}
    probas[nom] = y_proba
    print(f"{nom} — AUC : {auc:.3f} | CV : {cv.mean():.3f} ± {cv.std():.3f}")

## 3. Classification Report

In [ ]:
for nom, res in resultats.items():
    print(f"\n{'='*45}")
    print(f"  {nom}")
    print(f"{'='*45}")
    print(classification_report(y_test, res['y_pred'], target_names=['No Churn', 'Churn']))

## 4. Courbes ROC comparatives

In [ ]:
colors = ['#636EFA', '#EF553B', '#00CC96']
fig = go.Figure()

for (nom, y_proba), color in zip(probas.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = resultats[nom]['auc']
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr,
        mode='lines',
        name=f'{nom} (AUC = {auc:.3f})',
        line=dict(color=color, width=2)
    ))

# Diagonale aléatoire
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines',
    name='Aléatoire',
    line=dict(color='gray', width=1, dash='dash')
))

fig.update_layout(
    title='Courbes ROC — Comparaison des modèles',
    xaxis_title='Taux de faux positifs (FPR)',
    yaxis_title='Taux de vrais positifs (TPR)',
    legend=dict(x=0.6, y=0.1),
    width=700, height=500
)
fig.show()
fig.write_html(r'C:\Users\imene\churn-prediction\outputs\roc_curves.html')
print("✅ roc_curves.html exporté")

## 5. Matrice de confusion — XGBoost

In [ ]:
y_pred_xgb = resultats['XGBoost']['y_pred']
cm = confusion_matrix(y_test, y_pred_xgb)

fig = px.imshow(
    cm,
    text_auto=True,
    labels=dict(x='Prédit', y='Réel', color='Count'),
    x=['No Churn', 'Churn'],
    y=['No Churn', 'Churn'],
    color_continuous_scale='Blues',
    title='Matrice de confusion — XGBoost'
)
fig.show()

## 6. Sauvegarde du meilleur modèle

In [ ]:
best_model = modeles['XGBoost']
joblib.dump(best_model, r'C:\Users\imene\churn-prediction\src\xgb_churn_model.pkl')
joblib.dump(scaler, r'C:\Users\imene\churn-prediction\src\scaler.pkl')
joblib.dump(list(X.columns), r'C:\Users\imene\churn-prediction\src\feature_names.pkl')

print("✅ Modèle XGBoost sauvegardé")
print("✅ Scaler sauvegardé")
print("✅ Noms des features sauvegardés")